<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/AudioCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [2]:
# 1. Install the specialized directory splitting library
!pip uninstall -y split-folders && pip install -q split_folders


In [87]:
import torch
import torch.nn as nn
from datasets import load_dataset
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import splitfolders
import librosa
import librosa.display
import torch
from torch.utils.data import Dataset,DataLoader
import numpy as np
import torch.optim as optim
from tqdm import tqdm

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="UrbanSounds/urban_sounds_small",
    repo_type="dataset",
    local_dir="urban_sounds_small"
)

In [54]:
#Large Dataset
!kaggle datasets download -d chrisfilo/urbansound8k

Dataset URL: https://www.kaggle.com/datasets/chrisfilo/urbansound8k
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 5.61G/5.61G [01:23<00:00, 72.5MB/s]



In [57]:
!unzip -q urbansound8k.zip -d ./data_folder/

In [6]:
path = '/content/urban_sounds_small/urban_sounds_small'

In [58]:


# Define your input and output folders exactly as we mapped them out!
input_folder = '/content/data_folder' #'/content/urban_sounds_small/urban_sounds_small'
output_folder = '/content/urban_sounds8k_output'#'/content/urban_sounds_split'

# Execute the balanced physical split
splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(0.7, 0.3)
)

print("\n[SUCCESS] Folders split perfectly into train and test directories!")


Copying files: 8732 files [01:18, 111.77 files/s]


[SUCCESS] Folders split perfectly into train and test directories!


In [59]:
sourceRoot = output_folder
des_Root = '/content/urban_sounds8k_split_images'

In [96]:
class MelSpectrogramDataset(Dataset):
  def __init__(self,rootFolder) -> None:
    super().__init__()
    self.filePaths = []
    self.labels = []

    #Get sorted folder names to freeze class indexing
    self.class_names = sorted([f for f in os.listdir(rootFolder) if os.path.isdir(os.path.join(rootFolder,f))])

    for class_idx,class_name in enumerate(self.class_names):
      class_folder = os.path.join(rootFolder,class_name)
      for file_name in os.listdir(class_folder):
        if file_name.endswith('.pt'):
          self.filePaths.append(os.path.join(class_folder,file_name))
          self.labels.append(class_idx)

  def __len__(self):
    return len(self.filePaths)

  def __getitem__(self, idx):
    mel_tensor = torch.load(self.filePaths[idx])
    return mel_tensor,self.labels[idx]


In [97]:
train_data = MelSpectrogramDataset(rootFolder='/content/preprocessed/train')

val_data = MelSpectrogramDataset(rootFolder='/content/preprocessed/val')

In [98]:
print(len(train_data))

6109


In [104]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,num_workers=0,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,num_workers=0,pin_memory=True)

In [100]:
class CNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(1,32,3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(32,32,3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64,64,3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128,128,3),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(128,256,3,padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(256,256,3),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.flatten = nn.Flatten()

    self.fc = nn.Sequential(
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128,10)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)
    return x

In [101]:
model = CNN().to(device)

In [102]:
optimizer = optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

In [ ]:
epochs = 50
for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  Preloading_train = tqdm(train_loader,desc=f'Training_Epoch: {epoch+1}:')
  for images,label in Preloading_train:
    images = images.to(device)
    label = label.to(device)

    optimizer.zero_grad()

    with torch.amp.autocast('cuda'):
      outputs = model(images)
      loss = loss_fn(outputs,label)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item()
    pred = torch.argmax(outputs, dim=1)
    train_correct += (pred == label).sum().item()
    train_total += label.size(0)
  train_accuracy = train_correct / train_total
  train_average_loss = train_loss / len(train_loader)


  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    Preloading_val = tqdm(val_loader,desc=f'Validation_Epoch: {epoch+1}:')
    for images,label in Preloading_val:
      images = images.to(device)
      label = label.to(device)

      with torch.amp.autocast('cuda'):
        outputs = model(images)
        loss = loss_fn(outputs,label)

      val_loss += loss.item()
      pred = torch.argmax(outputs, dim=1)
      val_correct += (pred == label).sum().item()
      val_total += label.size(0)
    val_accuracy = val_correct / val_total
    val_average_loss = val_loss / len(val_loader)

  print('='*50)
  print(f'Epoch: {epoch+1}')
  print(f'Training_Accuracy: {train_accuracy} | Training_loss: {train_average_loss}')
  print(f'Validation_Accuracy: {val_accuracy} | Validation_loss: {val_average_loss}')
  print('='*50)


Validation_Epoch: 1:: 100%|██████████| 82/82 [00:01<00:00, 52.67it/s]


Epoch: 1
Training_Accuracy: 0.6071370109674251 | Training_loss: 1.353149011497098
Validation_Accuracy: 0.4235608082348456 | Validation_loss: 1.7249046535026737


Validation_Epoch: 2:: 100%|██████████| 82/82 [00:01<00:00, 47.65it/s]


Epoch: 2
Training_Accuracy: 0.6297266328367982 | Training_loss: 1.2963047857683991
Validation_Accuracy: 0.4292794510102936 | Validation_loss: 1.7026696481355807


Validation_Epoch: 3:: 100%|██████████| 82/82 [00:01<00:00, 53.12it/s]


...

Validation_Epoch: 41:: 100%|██████████| 82/82 [00:02<00:00, 38.94it/s]


Epoch: 41
Training_Accuracy: 0.9818300867572434 | Training_loss: 0.14953661998685117
Validation_Accuracy: 0.5440335493709493 | Validation_loss: 1.4951397347741011


Validation_Epoch: 42:: 100%|██████████| 82/82 [00:01<00:00, 51.08it/s]


Epoch: 42
Training_Accuracy: 0.9801931576362743 | Training_loss: 0.15092140537594
Validation_Accuracy: 0.5398398780022875 | Validation_loss: 1.5364480483822707


Training_Epoch: 43::  86%|████████▌ | 164/191 [00:05<00:00, 29.32it/s]

In [84]:
!rm -rf /content/preprocessed

In [94]:
%%writefile preprocess.py

import os
import torch
import librosa
import numpy as np
from tqdm import tqdm

def preprocess_split(source_root,save_root,target_sr=16000,duration=3,n_mels=128):
    os.makedirs(save_root, exist_ok=True)
    num_samples = target_sr * duration
    class_names = sorted([f for f in os.listdir(source_root) if os.path.isdir(os.path.join(source_root,f))])
    for class_name in class_names:
      class_folder = os.path.join(source_root,class_name)
      save_class_folder = os.path.join(save_root,class_name)
      os.makedirs(save_class_folder,exist_ok=True)
      files = os.listdir(class_folder)

      for file_name in tqdm(files,desc=class_name):
        if not file_name.endswith('.wav'):
          continue

        file_path = os.path.join(class_folder,file_name)
        audio, sr = librosa.load(file_path,sr=target_sr)

        # trim
        if len(audio) > num_samples:
          audio = audio[:num_samples]
        else:
          padding = num_samples - len(audio)
          audio = np.pad( audio,(0,padding))

        mel = librosa.feature.melspectrogram(y=audio,sr=sr,n_mels=n_mels)

        mel_db = librosa.power_to_db(mel,ref=np.max)

        mel_tensor = torch.tensor(mel_db,dtype=torch.float32)

        mel_tensor = mel_tensor.unsqueeze(0)

        save_path = os.path.join(save_class_folder,file_name.replace('.wav','.pt'))

        torch.save(mel_tensor,save_path)



if __name__ == "__main__":
  preprocess_split('/content/urban_sounds8k_output/train','/content/preprocessed/train')
  preprocess_split('/content/urban_sounds8k_output/val','/content/preprocessed/val')

Overwriting preprocess.py


In [95]:
!python preprocess.py

fold1: 100% 611/611 [00:19<00:00, 31.82it/s]
fold10: 100% 585/585 [00:13<00:00, 44.07it/s]
fold2: 100% 621/621 [00:14<00:00, 44.00it/s]
fold3: 100% 647/647 [00:14<00:00, 46.00it/s]
fold4: 100% 693/693 [00:16<00:00, 43.19it/s]
fold5: 100% 655/655 [00:14<00:00, 43.91it/s]
fold6: 100% 576/576 [00:12<00:00, 45.13it/s]
fold7: 100% 586/586 [00:13<00:00, 43.08it/s]
fold8: 100% 564/564 [00:13<00:00, 41.67it/s]
fold9: 100% 571/571 [00:13<00:00, 41.55it/s]
fold1: 100% 262/262 [00:05<00:00, 45.09it/s]
fold10: 100% 252/252 [00:05<00:00, 49.92it/s]
fold2: 100% 267/267 [00:04<00:00, 60.21it/s]
fold3: 100% 278/278 [00:06<00:00, 43.09it/s]
fold4: 100% 297/297 [00:05<00:00, 54.07it/s]
fold5: 100% 281/281 [00:06<00:00, 42.37it/s]
fold6: 100% 247/247 [00:04<00:00, 60.93it/s]
fold7: 100% 252/252 [00:04<00:00, 57.09it/s]
fold8: 100% 242/242 [00:05<00:00, 41.35it/s]
fold9: 100% 245/245 [00:04<00:00, 59.06it/s]
